Load the required libraries, the two trained pipelines saved from the training notebooks, and the raw dataset.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay,
)

log_reg_model = joblib.load('log_reg_model.joblib')
dec_tree_model = joblib.load('dec_tree_model.joblib')

df = pd.read_csv('dataset/cars_classification_dataset.csv')

Reproduce the exact same stratified train/test split used at training time by fixing `random_state=42` and `stratify=y`. Then obtain both hard predictions (`y_pred`) and class probabilities (`y_proba`) from each model on the untouched `X_test`. Probabilities are required for ROC-AUC and ROC curves.

In [ ]:
X = df.drop(columns=['car'])
y = df['car']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

y_pred_lr = log_reg_model.predict(X_test)
y_pred_dt = dec_tree_model.predict(X_test)

y_proba_lr = log_reg_model.predict_proba(X_test)
y_proba_dt = dec_tree_model.predict_proba(X_test)

Full classification report for Logistic Regression: precision, recall and F1-score per class, plus accuracy, macro average and weighted average. Rows are ordered from worst to best acceptability so the report matches the confusion matrix layout used later.

In [ ]:
report = classification_report(y_test, y_pred_lr, labels=['unacc', 'acc', 'good', 'vgood'])
print(report)

Same report for the Decision Tree, printed with the same class order so the two tables can be compared line by line.

In [ ]:
report = classification_report(y_test, y_pred_dt, labels=['unacc', 'acc', 'good', 'vgood'])
print(report)

Confusion matrix heatmaps for both models displayed side by side. Rows are the true classes, columns are the predicted classes, and the diagonal shows correct predictions.

In [ ]:
# One figure with two side-by-side subplots for direct visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left subplot: Logistic Regression confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr,
    labels=['unacc', 'acc', 'good', 'vgood'],
    cmap='Blues',
    ax=axes[0],
)
axes[0].set_title('Logistic Regression')

# Right subplot: Decision Tree confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_dt,
    labels=['unacc', 'acc', 'good', 'vgood'],
    cmap='Blues',
    ax=axes[1],
)
axes[1].set_title('Decision Tree')

plt.tight_layout()
plt.show()

Compute a macro-averaged multi-class ROC-AUC for each model using the One-vs-Rest strategy, then plot one ROC curve per class on a subplot per model. AUC measures how well the model ranks positive examples above negative ones and is independent of the decision threshold, which makes it robust on imbalanced datasets like this one. The dashed diagonal shows the performance of a random classifier as a reference.

In [ ]:
# predict_proba returns columns in the order stored in model.classes_ (alphabetical by default).
# We must binarize y_test in the same order so that column i of y_test_bin matches column i of y_proba.
class_labels_lr = list(log_reg_model.classes_)
class_labels_dt = list(dec_tree_model.classes_)

# One-hot encode y_test: each column becomes the binary ground truth for one One-vs-Rest problem
y_test_bin_lr = label_binarize(y_test, classes=class_labels_lr)
y_test_bin_dt = label_binarize(y_test, classes=class_labels_dt)

# Macro-averaged AUC across the 4 One-vs-Rest problems (one scalar per model)
auc_lr = roc_auc_score(y_test, y_proba_lr, multi_class='ovr', average='macro')
auc_dt = roc_auc_score(y_test, y_proba_dt, multi_class='ovr', average='macro')

print(f"Logistic Regression ROC-AUC (macro OvR): {auc_lr:.4f}")
print(f"Decision Tree      ROC-AUC (macro OvR): {auc_dt:.4f}")

# One subplot per model, with one ROC curve per class drawn on top
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Logistic Regression: draw one ROC curve per class using the One-vs-Rest binarization
for i, class_name in enumerate(class_labels_lr):
    RocCurveDisplay.from_predictions(
        y_test_bin_lr[:, i], y_proba_lr[:, i],
        name=class_name, ax=axes[0]
    )
# Dashed diagonal = random classifier reference
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_title(f'Logistic Regression (macro AUC = {auc_lr:.3f})')

# Decision Tree: same procedure
for i, class_name in enumerate(class_labels_dt):
    RocCurveDisplay.from_predictions(
        y_test_bin_dt[:, i], y_proba_dt[:, i],
        name=class_name, ax=axes[1]
    )
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_title(f'Decision Tree (macro AUC = {auc_dt:.3f})')

plt.tight_layout()
plt.show()

## Summary: Logistic Regression vs Decision Tree

The Decision Tree outperforms Logistic Regression on both headline metrics on the original test set.

| Metric | Logistic Regression | Decision Tree |
|--------|---------------------|---------------|
| Accuracy | 0.81 | 0.85 |
| Macro F1-score | 0.75 | 0.80 |
| Macro ROC-AUC (OvR) | 0.9484 | 0.9692 |

Both models reach very high ROC-AUC scores, which means their predicted probabilities rank positive examples above negative ones almost perfectly. The gap between accuracy (0.81) and AUC (0.95) for Logistic Regression indicates that the model actually ranks the classes well but the default 0.5 decision threshold is not optimal for this dataset.

The Decision Tree wins on macro F1-score as well as on macro ROC-AUC. This matches the nature of the data: car acceptability is driven by strict conditional rules (for example `persons=2` or `safety=low` immediately force the `unacc` class), and a tree captures these non-linear boundaries natively, while a linear model can only approximate them.

Decision Tree is the better baseline for this problem.